<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/VITCLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import tensorflow as tf
from tensorflow.keras import layers

In [4]:
class TransformerBlock(layers.Layer):
  def __init__(self,num_heads,embed_dim,ff_dim,rate=0.1) -> None:
    super().__init__()
    self.att = layers.MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)

    self.fft = tf.keras.Sequential([
        layers.Dense(ff_dim,activation='relu'),
        layers.Dense(embed_dim)
    ])

    self.layer_norm1 = layers.LayerNormalization(epsilon=1e-6)
    self.layer_norm2 = layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = layers.Dropout(rate=rate)
    self.dropout2 = layers.Dropout(rate=rate)

  def call(self,input):
    #Brick3:Multihead
    attention_output = self.att(query=input,key=input,value=input)
    attention_output = self.dropout1(attention_output)

    #Brick4:Blue
    merged1 = self.layer_norm1(input + attention_output)

    #Brick3:MLP
    fft_output = self.fft(merged1)
    fft_output = self.dropout2(fft_output)

    return self.layer_norm2(merged1 + fft_output)


In [7]:
#Patch Embedding
class PatchEmbedding(layers.Layer):
  def __init__(self,embed_dim,image_size,patch_size) -> None:
    super().__init__()
    self.num_patchs = (image_size // patch_size) ** 2
    self.patchEmbed = layers.Dense(embed_dim)
    self.positionEmbed = layers.Embedding(input_dim=self.num_patchs,output_dim=embed_dim)

  def call(self,patch):
    position_range = tf.range(start=0,limit=self.num_patchs,delta=1)
    patch_output = self.patchEmbed(patch)
    position_output = self.positionEmbed(position_range)

    return patch_output + position_output

In [9]:
#VIT with flatten (which basically take all the features and numbers and make a giant number)
class VisionTransformer(tf.keras.Model):
  def __init__(self,patch_size,embed_dim,image_size,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.patchEmbed = PatchEmbedding(embed_dim=embed_dim,image_size=image_size,patch_size=patch_size)

    self.transform_layers = [
        TransformerBlock(num_heads=num_heads,embed_dim=embed_dim,ff_dim=ff_dim)
        for _ in range(num_layers)
    ]

    self.flatten = layers.Flatten()
    self.output_layer = layers.Dense(num_classes,activation='softmax')

  def call(self,images):
    patchs = self.extract_images(images)
    x = self.patchEmbed(patchs)

    for transform_layer in self.transform_layers:
      x = transform_layer(x)

    x = self.flatten(x)
    return self.output_layer(x)

  def extract_images(self,images):
    batch_size = tf.shape(images)[0]

    patchs = tf.image.extract_patches(
        images=images,
        sizes=[1,patch_size,patch_size,1],#1 means batch size (one image at a time), patch size are hight width and last 1 means take every step without miss, always use 1 here
        strides=[1,patch_size,patch_size,1],
        rates=[1,1,1,1],#here 1,1,1,1 means pick every pixal of image inside every patch
        padding='SAME'
    )
    output = tf.reshape(patchs,[batch_size,-1,patch_size * patch_size *3])
    return output


In [11]:
#VIT (Here we use global average pool which takes average from each and make it to our embed_dims features)
class VisionTransformer(tf.keras.Model):
  def __init__(self,patch_size,embed_dim,image_size,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.patchEmbed = PatchEmbedding(embed_dim=embed_dim,image_size=image_size,patch_size=patch_size)

    self.transform_layers = [
        TransformerBlock(num_heads=num_heads,embed_dim=embed_dim,ff_dim=ff_dim)
        for _ in range(num_layers)
    ]

    self.pool = layers.GlobalAveragePooling1D()
    self.output_layer = layers.Dense(num_classes,activation='softmax')

  def call(self,images):
    patchs = self.extract_images(images)
    x = self.patchEmbed(patchs)

    for transform_layer in self.transform_layers:
      x = transform_layer(x)

    x = self.pool(x)
    return self.output_layer(x)

  def extract_images(self,images):
    batch_size = tf.shape(images)[0]

    patchs = tf.image.extract_patches(
        images=images,
        sizes=[1,patch_size,patch_size,1],#1 means batch size (one image at a time), patch size are hight width and last 1 means take every step without miss, always use 1 here
        strides=[1,patch_size,patch_size,1],
        rates=[1,1,1,1],#here 1,1,1,1 means pick every pixal of image inside every patch
        padding='SAME'
    )
    output = tf.reshape(patchs,[batch_size,-1,patch_size * patch_size *3])
    return output


Here first we create a cls token that will minitor the updates inside the transformer block. here in cls token, we use 1,1 for shape that means create 1 copy of every image(batch) and another 1 is sequence means:

Remember that a Transformer thinks of an image as a Sentence.The 196 image patches are like 196 words.The [CLS] token is like adding 1 extra word to the start of that sentence.
so that 1 extra word will take all the information, and in the end we ask that word to give us what it has info about that image and that we use it to pass for our final dense layer instead of FLATTEN!!!

In [10]:
#VIT (Here we use CLS which keep and eye on transformer via token and in the end we dont have to summarize everything, we just take information from the agent we put inside the transformer to look for)
class VisionTransformer(tf.keras.Model):
  def __init__(self,patch_size,embed_dim,image_size,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.patchEmbed = PatchEmbedding(embed_dim=embed_dim,image_size=image_size,patch_size=patch_size)

    self.transform_layers = [
        TransformerBlock(num_heads=num_heads,embed_dim=embed_dim,ff_dim=ff_dim)
        for _ in range(num_layers)
    ]

    self.cls_token = self.addWeight(
        name='cls_token',
        shape=(1,1,embed_dim),
        initializer='random_normal',
        trainable=True
    )

    self.output_layer = layers.Dense(num_classes,activation='softmax')

  def call(self,images):
    patchs = self.extract_images(images)
    x = self.patchEmbed(patchs)

    # 1. Duplicate CLS token for the whole batch
    # Turns (1, 1, 128) into (batch, 1, 128)
    batch_size = tf.shape(x)[0]
    cls_broadcasted = tf.cast(tf.broadcast_to(self.cls_token, [batch_size, 1, self.embed_dim]), dtype=x.dtype)
    # 2. Glue it to the front!
    # Turns (batch, 196, 128) into (batch, 197, 128)
    x = tf.concat([cls_broadcasted, x], axis=1)

    for transform_layer in self.transform_layers:
      x = transform_layer(x)

    # 4. THE EXIT: Take only the first token (The Reporter)
    # Shape: (batch, 128)
    x = x[:, 0]
    return self.output_layer(x)

  def extract_images(self,images):
    batch_size = tf.shape(images)[0]

    patchs = tf.image.extract_patches(
        images=images,
        sizes=[1,patch_size,patch_size,1],#1 means batch size (one image at a time), patch size are hight width and last 1 means take every step without miss, always use 1 here
        strides=[1,patch_size,patch_size,1],
        rates=[1,1,1,1],#here 1,1,1,1 means pick every pixal of image inside every patch
        padding='SAME'
    )
    output = tf.reshape(patchs,[batch_size,-1,patch_size * patch_size *3])
    return output